# **UPLOAD cleaned_transcripts.csv first before running all cells! **

In [4]:
# --- CODE TF-IDF & SVM HOÀN CHỈNH (ALL-IN-ONE) ---
import pandas as pd
import numpy as np
import os
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import f1_score, recall_score

# 1. LOAD DỮ LIỆU
current_dir = os.getcwd()
csv_path = os.path.abspath(os.path.join(current_dir, '..', '..', 'preprocessing', 'clean_compiled_transcripts.csv'))
test_file_path = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data', 'test_split_Depression_AVEC2017.csv'))

print("🚀 BẮT ĐẦU QUY TRÌNH TF-IDF...")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    if 'Text' in df.columns: 
        df.rename(columns={'Text': 'Transcript', 'PHQ8_Binary': 'PHQ_Binary'}, inplace=True)
    df['Transcript'] = df['Transcript'].fillna('')
    df.set_index('Participant_ID', inplace=True)
    print(f"✅ Load xong {len(df)} mẫu dữ liệu.")
else:
    print("❌ LỖI: Không tìm thấy file CSV!")

# 2. TẠO TF-IDF
print("⏳ Đang tạo vector TF-IDF...")
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=3)
X = tfidf.fit_transform(df['Transcript']).toarray()
y = df['PHQ_Binary']

# 3. CHIA TẬP TRAIN/TEST & UNDERSAMPLING
def prepare_data(X, y, testfile):
    test_participants = pd.read_csv(testfile).iloc[:, 0].values
    X_train, X_test, y_train, y_test = [], [], [], []
    
    for i in range(y.shape[0]):
        p_id = y.index[i]
        if p_id in test_participants:
            X_test.append(X[i]); y_test.append(y.iloc[i])
        else:
            X_train.append(X[i]); y_train.append(y.iloc[i])
            
    # Undersampling
    random.seed(42)
    neg = [i for i, val in enumerate(y_train) if val == 0]
    pos = [i for i, val in enumerate(y_train) if val == 1]
    n_samples = min(len(neg), len(pos))
    final_idx = random.sample(neg, n_samples) + random.sample(pos, n_samples)
    random.shuffle(final_idx)
    
    return np.array([X_train[i] for i in final_idx]), np.array([y_train[i] for i in final_idx])

X_train, y_train = prepare_data(X, y, test_file_path)
print(f"✅ Dữ liệu train sau khi cân bằng: {X_train.shape}")

# 4. TRAIN MODEL SVM (n_jobs=1 để fix lỗi Windows)
print("🧠 Đang huấn luyện SVM (Grid Search)...")
params = [{'kernel': ['linear'], 'C': [1, 10, 100]}, {'kernel': ['rbf'], 'gamma': [1e-3, 1e-4], 'C': [1, 10, 100]}]
svm = GridSearchCV(SVC(), params, cv=5, scoring='f1', n_jobs=1)
svm.fit(X_train, y_train)

print("-" * 30)
print(f"🏆 Best F1-Score: {svm.best_score_:.4f}")
print(f"🔧 Best Params: {svm.best_params_}")
print("-" * 30)

🚀 BẮT ĐẦU QUY TRÌNH TF-IDF...
✅ Load xong 142 mẫu dữ liệu.
⏳ Đang tạo vector TF-IDF...
✅ Dữ liệu train sau khi cân bằng: (84, 1000)
🧠 Đang huấn luyện SVM (Grid Search)...
------------------------------
🏆 Best F1-Score: 0.6336
🔧 Best Params: {'C': 1, 'kernel': 'linear'}
------------------------------
